# 02. Nested data

**Tier:** Deep. **Why you'd use this:** A folder tree or a nested list of numbers is not a flat loop. You enter a node, handle children, then leave — and the print order of enter and leave shows that shape.

**After this lesson you can:** follow the order of enter and leave prints on a tree; sum a nested list and a tree of tuples.
About 25 minutes. **Uses:** [Base case](01-base-case-lesson.ipynb), [Lists](../04-collections/01-lists-lesson.ipynb)


## Quick review

1. What is a base case for?
2. What happens to the call stack as recursive calls go deeper?

<details><summary>Answers</summary>

It stops the recursion with a direct answer. The stack grows with each call and shrinks on the way back up.
</details>


## The idea

Some data is made of smaller pieces that have the same shape as the whole. A list of lists is that shape. A tree is that shape too: one value, then a sequence of smaller trees.

A node here is a pair `(value, children)`. `children` is a tuple of nodes. A leaf is a node whose children tuple is empty. Depth is how many nodes you pass from the root down to a leaf, counting both.

The function that understands one node understands every node. It handles the value it is holding, then calls itself on each child. The call stack follows the nesting. A loop inside the function walks siblings. The self-call walks down.

<details><summary>Explain more</summary>

Siblings are the children of one parent. Only one sibling's frame is alive at a time, because the `for` loop starts the next child after the previous call returns. A bush of a thousand leaves can sit at depth 2. A stick of four nodes needs four frames. The shape of the data, not the number of values, decides the stack.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This tree is not a class. It is a tuple so the lesson can stay on data you already have. A class with a list of children is the same walk, once you have classes.
</details>


## See it work

A file of sensor groups can nest. You want the total of every number, including numbers buried in inner lists. `sum` stops at the first level and cannot add a list to an int. The walk has to decide, for each item, whether it is a list or a number.


In [1]:
def nested_sum(items: list) -> int:
    # 1. Start from nothing at this level.
    total = 0
    for item in items:
        # 2. A list is the same problem, one level down.
        if isinstance(item, list):
            total += nested_sum(item)
        # 3. A number is a leaf. Add it.
        else:
            total += item
    return total


print(nested_sum([1, [2, 3], 4]))
print(nested_sum([]))
print(nested_sum([[], [5]]))

10
0
5


The same walk works on the tuple tree. The node is not a list of mixed values. It is a value plus children, so you always add the value, then visit each child. There is no `isinstance` test, because every child is a node.


In [2]:
def tree_sum(node: tuple) -> int:
    # 1. This node's own value.
    value, children = node
    total = value
    # 2. The same problem on each child.
    for child in children:
        total += tree_sum(child)
    return total


sample = (2, ((1, ()), (4, ((3, ()),))))
print(tree_sum(sample))
print(tree_sum((7, ())))

10
7


| step | action | values still on the stack |
| --- | --- | --- |
| 1 | enter 2 | 2 |
| 2 | enter 1, leaf, add 1 | 2, 1 |
| 3 | leave 1 | 2 |
| 4 | enter 4 | 2, 4 |
| 5 | enter 3, leaf, add 3 | 2, 4, 3 |
| 6 | leave 3, 4's total is 7 | 2, 4 |
| 7 | leave 4 | 2 |
| 8 | leave 2, total is 10 | none |

```
2
├── 1
└── 4
    └── 3
```


## Check yourself

Answer these from the example above before you run the cell.


1. Why is `leave 1` the last line in the example above, not the second?
2. Why does the bush use fewer frames than the stick when both hold four nodes?
3. What does `nested_sum` do with an empty inner list, and why is that not an error?

<details><summary>Answers</summary>

1. `leave` is after the loop. The loop calls `walk` on every child and waits. The root cannot reach its own `leave` until both children have returned.
2. The bush's parent frame visits one leaf, gets the return, then visits the next leaf. Two frames is the peak. The stick has no siblings, so each node waits on the single node below it, and all four frames exist together.
3. The `for` over an empty list runs zero times and adds nothing. The inner call returns 0. Adding 0 is the same as skipping that group.
</details>


In [3]:
def walk(node):
    value, children = node
    print("enter", value)
    for child in children:
        walk(child)
    print("leave", value)


walk((1, ((2, ()), (3, ()))))

enter 1
enter 2
leave 2
enter 3
leave 3
leave 1


<details><summary>Check your answers</summary>

- `enter 1` runs before either child. The root frame is created first.
- `enter 2`, then `leave 2`. Node 2 is a leaf, so its loop does not call `walk`.
- `enter 3` and `leave 3` happen only after node 2 has returned. The root's loop is still on its first frame's `for`.
- `leave 1` is last. The root's print after the loop waited for every child.
</details>


## Why it works this way

The stack is a picture of the path from the root to the node you are in. When `tree_sum` is inside node 3, the frames for 2, 4, and 3 are all alive. When it moves to the next sibling, 3's frame is gone before 4's loop continues. Siblings do not sit on the stack together. Cousins do not either. Only the ancestors of the current node are waiting.

That is why a wide tree is cheap in memory and a skinny tree is not. Four nodes in a row, each the only child of the one above, need four frames at the deepest moment. Four nodes as one parent and three leaves need two: the parent, and one leaf. The `for` loop reuses the parent's frame for every child. The number of values can be large while the depth stays small. Time is still one visit per node, a constant amount of work each, so a tree of n nodes takes time proportional to n. Extra memory is proportional to the depth, not to n, unless the tree is a single path.

A flat loop cannot name "the rest of this child" without storing the path somewhere. Recursion stores that path in the frames you were going to have anyway. The design reason is the match between the data and the call: a node is a value plus nodes, and the function is "this value, plus the function on each part".

`sum` on one level is the wrong tool once a list contains a list. It adds items with `+`. A list plus an int raises. The recursive function does not add the inner list. It adds the number that the inner call returns.

The cell below measures the deepest number of live frames on a stick and on a bush with the same four nodes. The stick peaks at 4. The bush peaks at 2. Same amount of data, different stack.


In [4]:
def peak_frames(node: tuple) -> int:
    def visit(current: tuple, live: int) -> int:
        _value, children = current
        highest = live
        for child in children:
            highest = max(highest, visit(child, live + 1))
        return highest

    return visit(node, 1)


stick = (1, ((2, ((3, ((4, ()),)),)),))
bush = (1, ((2, ()), (3, ()), (4, ())))
print(peak_frames(stick), peak_frames(bush))

4 2


## When to use it

Use this walk when the input nests and every level is the same kind of job: totals in grouped readings, the size of a folder of folders, an expression made of smaller expressions. The tuple tree and the nested list are two spellings of that job.

Use an ordinary loop when the list is flat. Recursion adds frames and an `isinstance` test you do not need. Use an explicit stack, from the next lesson, when a user can nest data thousands of levels deep. A stick of that length will hit the recursion limit even though the algorithm is fine.

`sum` is the right tool for a list of numbers and the wrong tool for a list of lists. Flattening first, then summing, is another option when you need the flat sequence for something else. If you only need the total, the walk above does not build that sequence.


## Common mistake

Adding `child[0]` adds the child's own value and skips the child's descendants. The total looks plausible. It is short by every grandchild. Node 3 never gets added.


In [5]:
def tree_sum_wrong(node: tuple) -> int:
    value, children = node
    total = value
    for child in children:
        total += child[0]
    return total


sample = (2, ((1, ()), (4, ((3, ()),))))
print(tree_sum_wrong(sample))

7


The print is `7`, which is `2 + 1 + 4`. The `3` under `4` was never visited. `child[0]` is a number, not a call, so the frame for 4 does not create a frame for 3. The fix calls `tree_sum` on the child. That call adds the child's value and everything under it, then returns one number.


In [6]:
def tree_sum(node: tuple) -> int:
    value, children = node
    total = value
    for child in children:
        total += tree_sum(child)
    return total


sample = (2, ((1, ()), (4, ((3, ()),))))
print(tree_sum(sample))

10


## Try it

Change `walk` so it returns the values in leave order instead of printing them. For `(1, ((2, ()), (3, ())))` the list should be `[2, 3, 1]`. Why is the root last, and what list do you get if you record the value before the loop instead of after it?


## Recap

- A nested list needs a test: list means "call again", number means "add it".
- A tuple node is a value plus children. Add the value, then call on each child.
- Live frames follow the path to the current node. Width is cheap. A long chain is not.

Next: [practice](02-structures-practice.ipynb).
